# 04 - Loss functions, composite loss, and the agents (Part C)

Three jobs here:
1. **Loss sweep:** train each model with each of the 6 losses, identical settings, and compare on validation.
2. **Composite-weight agent:** Optuna searches the weights lambda of the composite loss, using only train+val.
3. **Threshold agent:** exhaustive search for the best confidence threshold, again only on validation.

Every trial is logged to a CSV in `results/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q segmentation-models-pytorch transformers optuna thop joblib
print('installs done')

In [ ]:
import sys, os
import numpy as np
sys.path.insert(0, '/content/drive/MyDrive/coco_seg/project/code')
import config as C
import data, engine, losses, models, agent_search, report_utils
C.ensure_dirs()
print('GPU:', __import__('torch').cuda.get_device_name(0) if __import__('torch').cuda.is_available() else 'NONE - Runtime > Change runtime type > T4 GPU')
frames = data.load_frames()
class_weights = report_utils.load_result('class_weights')
assert class_weights, 'Run 01_dataset_stats.ipynb first'
CW = class_weights['median_frequency_weights']
print('class weights loaded')

### Part 1: the loss sweep
Six losses, two models, identical epochs. **This is the longest step**.


In [ ]:
SWEEP_EPOCHS = C.EPOCHS_SWEEP     # identical for every loss = fair
for name in C.MODELS:
    train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True)
    val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
    results = report_utils.load_result(f'loss_sweep_{name}') or {}
    for loss_name in C.LOSSES:
        if loss_name in results:
            print(f'[skip] {name}/{loss_name} already done')
            continue
        print(f'=== {name} with {loss_name} ===')
        model = models.build_model(name)
        history = engine.train(model, f'{name}_sweep_{loss_name}', train_loader, val_loader,
                               losses.build_loss(loss_name, C.NUM_CLASSES, CW),
                               epochs=SWEEP_EPOCHS, seed=C.SEEDS[0])
        val_metrics = engine.evaluate(model, val_loader)
        results[loss_name] = {'history': history, 'val': val_metrics}
        report_utils.save_result(f'loss_sweep_{name}', results)   # save after every loss
        del model
        __import__('torch').cuda.empty_cache()
    print(name, 'sweep done:', {k: round(v['val']['miou'], 4) for k, v in results.items()})

### Part 2: composite-loss weight agent (Optuna)

The composite combines three loss families: `ce` (distribution), `dice` (region), `lovasz` (metric). The agent searches the weights.

In [ ]:
COMPONENTS = ['ce', 'dice', 'lovasz']   # one per loss family; you can change this (see MAKE_IT_YOURS.md)
for name in C.MODELS:
    if report_utils.load_result(f'agent_weight_choice_{name}'):
        print(f'[skip] weight agent for {name} already done')
        continue
    def make_model_and_loaders():
        m = models.build_model(name)
        tl = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name],
                             augment=True, subset=C.AGENT_TRAIN_SUBSET)
        vl = data.get_loader(frames['val'], 'val', batch_size=C.BATCH_SIZE[name], subset=300)
        return m, tl, vl
    choice = agent_search.composite_weight_agent(name, COMPONENTS, make_model_and_loaders)
    print(name, 'chosen weights:', choice['chosen_weights'],
          'val mIoU (proxy):', round(choice['chosen_val_miou'], 4))

### Part 3: train the composite loss fully, then run the threshold agent

The chosen weights get a full training run, then the threshold agent does an exhaustive search over confidence thresholds on validation. Compare its choice against the default (0.0)

In [ ]:
for name in C.MODELS:
    choice = report_utils.load_result(f'agent_weight_choice_{name}')
    train_loader = data.get_loader(frames['train'], 'train', batch_size=C.BATCH_SIZE[name], augment=True)
    val_loader   = data.get_loader(frames['val'],   'val',   batch_size=C.BATCH_SIZE[name])
    model = models.build_model(name)
    comp = losses.CompositeLoss(
        {k: losses.build_loss(k, C.NUM_CLASSES, CW) for k in choice['components']},
        choice['chosen_weights'])
    history = engine.train(model, f'{name}_composite', train_loader, val_loader,
                           comp, epochs=C.EPOCHS_MAIN, seed=C.SEEDS[0])
    __import__('torch').save(model.state_dict(), os.path.join(C.CKPT_DIR, f'{name}_composite_final.pt'))
    sweep = engine.threshold_sweep(model, val_loader)
    tchoice = agent_search.threshold_agent(sweep, name)
    val_metrics = engine.evaluate(model, val_loader, threshold=tchoice['chosen_threshold'])
    val_metrics['seconds_per_epoch'] = float(np.mean([h['epoch_seconds'] for h in history]))
    report_utils.save_result(f'composite_main_{name}', {'history': history, 'val': val_metrics,
                                                        'weights': choice['chosen_weights'],
                                                        'threshold': tchoice['chosen_threshold']})
    print(f"{name}: composite val mIoU {val_metrics['miou']:.4f} at threshold {tchoice['chosen_threshold']}",
          f"(default 0.0 gives {tchoice['default_val_miou']:.4f})")

**agent logs:** `results/agent_weight_trials_*.csv` and `results/agent_threshold_trials_*.csv`